# 09 · Modelo final para la API y el tablero

Entrena y exporta el modelo elegido: **LightGBM con pérdida `l1`** (cuaderno 04). Para cada caso se entrenan tres modelos: el **precio de referencia** y los límites del **rango sugerido** (percentiles 7,5 y 92,5, cuaderno 07).

| `usar_coordenadas` | Conjunto | `learning_rate` | Archivo |
|---|---|---|---|
| `False` | `v2` | 0,02 | `api/modelos/modelo_v2.joblib` |
| `True` | `v2_geo` (v2 + latitud y longitud) | 0,01 | `api/modelos/modelo_v2_geo.joblib` |

La configuración del entrenamiento está en `src/bogoprice/modelo_final.py`. Los modelos se guardan en la carpeta **`api/`**, que funciona sola: `api/predictor.py` recibe los datos crudos de Inside Airbnb, hace toda la transformación de variables y predice, sin depender del paquete `bogoprice`. Las corridas van al experimento `bogoprice-modelo-final` de MLflow.

In [ ]:
import pandas as pd

from bogoprice import modelo_final
from bogoprice.experimentos import preparar_datos
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"
iniciar_mlflow("bogoprice-modelo-final")

## 1. Validación

Antes del entrenamiento final se verifica la configuración con el mismo esquema de los demás cuadernos: entrenar con junio y evaluar con septiembre. Estas métricas quedan guardadas dentro del paquete del modelo, para mostrarlas en el panel de monitoreo.

In [ ]:
datos = preparar_datos()
validacion = {}
for usar_coordenadas in [False, True]:
    version = modelo_final.version_de(usar_coordenadas)
    m = modelo_final.validar(usar_coordenadas, datos)
    validacion[version] = m
    registrar_corrida(
        f"validacion_{version}",
        {"modelo": "LightGBM", "conjunto": version, "entrenamiento": "2026-06",
         "evaluacion": "2026-09", "percentiles": str(modelo_final.PERCENTILES),
         **modelo_final.PARAMETROS_PRECIO[version]},
        m, tags={"autor": AUTOR, "conjunto": version, "etapa": "validacion"})

pd.DataFrame(validacion).T.round(2)

## 2. Entrenamiento final

Se entrena con **junio y septiembre juntos**, como se definió en el alcance. Si un anuncio aparece en las dos capturas, se usa su observación más reciente. El límite de atípicos, las localidades habilitadas y las tipologías se recalculan con los datos unidos.

In [ ]:
df_final, info = modelo_final.preparar_entrenamiento_final()
print(f"Anuncios: {info['n_anuncios']} | P99: ${info['limite_p99']:,.0f} | "
      f"localidades: {len(info['localidades'])} | tipologías: {len(info['tipologias'])}")

rutas = {}
for usar_coordenadas in [False, True]:
    version = modelo_final.version_de(usar_coordenadas)
    paquete = modelo_final.entrenar_paquete(usar_coordenadas, df_final, info, validacion[version])
    ruta = modelo_final.guardar_paquete(paquete)
    rutas[version] = ruta
    registrar_corrida(
        f"final_{version}",
        {"modelo": "LightGBM", "conjunto": version, "capturas": "+".join(info["capturas"]),
         "n_anuncios": info["n_anuncios"], **modelo_final.PARAMETROS_PRECIO[version]},
        {f"validacion_{k}": v for k, v in validacion[version].items()},
        pipeline=paquete["modelo"], artefactos=[str(ruta)],
        tags={"autor": AUTOR, "conjunto": version, "etapa": "final"})
    print(f"{version:7s} guardado en {ruta} ({ruta.stat().st_size / 1e6:.1f} MB)")

# Versiones exactas de las librerías para la API
print("Requisitos de la API:", modelo_final.escribir_requirements_api(paquete))

## 3. Uso de la carpeta `api/` con datos crudos

`api/predictor.py` se importa directamente desde la carpeta `api/`, sin el paquete `bogoprice`. Recibe filas del CSV crudo de Inside Airbnb, tal como vienen, y devuelve el precio de referencia y el rango sugerido.

In [ ]:
import sys
sys.path.insert(0, str(modelo_final.CARPETA_API))
from predictor import Predictor

from bogoprice.data_loader import cargar_y_limpiar

predictor = Predictor()
crudo = cargar_y_limpiar("2026-09")          # CSV crudo de septiembre

columnas_crudas = ["id", "neighbourhood_cleansed", "room_type", "property_type", "accommodates",
                   "bedrooms", "beds", "bathrooms_text", "amenities", "minimum_nights",
                   "latitude", "longitude"]
muestra = crudo[crudo["neighbourhood_cleansed"].isin(["Chapinero", "Usaquen", "Teusaquillo"])]
muestra = muestra.sample(5, random_state=1)

pred = predictor.predecir(muestra[columnas_crudas], usar_coordenadas=True)
pd.concat([muestra[["id", "neighbourhood_cleansed", "room_type", "accommodates", "price"]], pred],
          axis=1)

In [ ]:
# Un anuncio como diccionario (lo que enviaría el formulario del tablero)
anuncio = {
    "neighbourhood_cleansed": "Chapinero",
    "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
    "accommodates": 2,
    "bedrooms": 1,
    "beds": 1,
    "bathrooms_text": "1 bath",
    "amenities": ["Wifi", "Kitchen", "Elevator", "Dedicated workspace", "Hot water", "TV"],
    "minimum_nights": 2,
    "latitude": 4.6486,     # solo se usan con usar_coordenadas=True
    "longitude": -74.0603,
}

pd.DataFrame({
    "sin coordenadas": predictor.predecir(anuncio, usar_coordenadas=False),
    "con coordenadas": predictor.predecir(anuncio, usar_coordenadas=True),
})

In [ ]:
# Aporte de cada variable al precio (gráfico de contribución del tablero)
explicacion = predictor.explicar(anuncio, usar_coordenadas=True)
print(f"Precio base: ${explicacion.attrs['precio_base_cop']:,.0f}  ->  "
      f"precio de referencia: ${explicacion.attrs['precio_referencia_cop']:,.0f}")
explicacion.head(10)

## 4. Verificación: la API transforma igual que el entrenamiento

Se toman los anuncios de septiembre, se transforman por los dos caminos (el del entrenamiento, con `features.py` y `variables_v2.py`, y el de `api/predictor.py`) y se comparan las predicciones. Deben ser idénticas.

In [ ]:
import numpy as np
from bogoprice import variables_v2
from bogoprice.features import construir_features

crudo_validos = crudo[crudo["id"].isin(datos.test["id"])]
for usar_coordenadas in [False, True]:
    p = predictor.paquete(usar_coordenadas)
    entrenamiento = variables_v2.agregar_variables_v2(construir_features(crudo_validos),
                                                      p["tipologias"])[p["columnas"]]
    X_api, motivo = predictor.preparar(crudo_validos, usar_coordenadas)
    ok = motivo.eq("")
    diferencia = np.abs(p["modelo"].predict(entrenamiento[ok]) - p["modelo"].predict(X_api[ok])).max()
    print(f"{p['version']:7s} anuncios: {ok.sum()} | diferencia máxima en predicción: {diferencia:.2e}")
    assert diferencia < 1e-9, "La transformación de la API no coincide con la del entrenamiento"
print("La transformación de api/predictor.py es idéntica a la del entrenamiento.")